# EMSR Data Preparation for STURM-Flood Inference (Sentinel-1, SAR)

This notebook prepares Copernicus EMS flood data for inference with the STURM-Flood U-Net model.

**Workflow overview:**
1. **Mount Drive & Select Radar** - Load preprocessed Sentinel-1 GeoTIFF from Google Drive
2. **Upload EMS Vectors** - Upload ZIP file from Copernicus Emergency Management Service
3. **Prepare AOI** - Clean the Area of Interest by removing "Not Analysed" zones
4. **Clip Radar** - Crop the radar image to match the clean AOI boundary
5. **Create Ground Truth** - Rasterize EMS vector polygons to create labeled mask
6. **Normalize & Tile** - Apply STURM normalization and split into 128x128 tiles
7. **Filter & Export** - Remove invalid tiles and create metadata CSV


**STURM-Flood class mapping:**
- Class 0: Background/Land
- Class 1: Flooded areas (observed flood extent)
- Class 2: Rivers and streams (permanent water - linear features)
- Class 3: Open water (sea/ocean - for coastal events)
- Class 4: Reservoirs (if present)
- Class 5: Lakes (permanent water - polygon features)
- Class 99: No data (outside analyzed area)

## 1. Setup and Dependencies

Install and import required Python packages for geospatial processing.

In [ ]:
# ============================================================
# CONFIGURATION — change only this section
# ============================================================
# BASE is the parent folder in your Google Drive where the
# project's data and outputs live.
#
# Set BASE here, then mount your Google Drive in the cell below before running the rest.
# OUTPUT_DATASET_DIR is the folder where prepared tiles will
# be saved (one subfolder Sentinel1/S1 for radar tiles, one
# Sentinel1/Floodmaps for ground truth tiles).
# ============================================================

BASE = '/content/drive/MyDrive/Project_data_Stefan_Edvinsson'
OUTPUT_DATASET_DIR = f'{BASE}/Dataset_goteborg'   # change to e.g. Dataset_karlstad for other area


In [ ]:
# Install geospatial packages (required in Google Colab)
# - geopandas: For reading/writing vector data (shapefiles)
# - rasterio: For reading/writing raster data (GeoTIFF)
# - fiona: Backend for geopandas file I/O
# - shapely: For geometric operations (clip, union, difference)
# - pyproj: For coordinate reference system transformations

!pip install geopandas rasterio fiona shapely pyproj --quiet

print("✓ Dependencies installed!")

In [ ]:
# Import all required libraries
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features          # For rasterizing vector data
from rasterio.mask import mask         # For clipping rasters with polygons
from rasterio.windows import Window    # For reading raster subsets (tiles)
from pathlib import Path
from shapely.ops import unary_union    # For merging multiple polygons
from google.colab import drive, files  # For Google Drive integration
import shutil
import zipfile
import ipywidgets as widgets           # For interactive file picker
from IPython.display import display, clear_output

print("✓ Libraries imported successfully!")

## 2. Configuration

Set parameters for the specific EMS event and processing options.

**Important settings:**
- `target_crs`: Must match your radar data CRS (UTM zone for the area)
- `tile_size`: STURM-Flood uses 128x128 pixel tiles
- `max_nodata_pct`: Set to 0 to only keep tiles fully within AOI

In [ ]:
# ============================================================================
# CONFIGURATION - Edit these values for your specific EMS event
# ============================================================================

CONFIG = {
    # ----- Event Metadata -----
    # These are used for naming output files and metadata CSV
    'ems_code': 'EMSR427',
    'event_type': 'Riverine flood',
    'country': 'Sweden',
    'aoi_code': 'AOI04',                                                # Göteborg AOI within EMSR427 activation
    'sentinel_date': '2020-02-22 16:53',                                # Sentinel-1 acquisition matches current flood layer
    'floodmap_date': '2020-02-22 16:53',                                # EMS "Flooded Area" timestamp (not "Previous")


    # ----- Coordinate Reference System -----
    # UTM zone should match your study area:
    # - Sweden (Västra Götaland): EPSG:32633 (UTM 33N)
    # - Central Europe: EPSG:32632 (UTM 32N)
    # - UK/Ireland: EPSG:32630 (UTM 30N)
    'target_crs': 'EPSG:32633',

    # ----- Tile Parameters -----
    # STURM-Flood model expects 128x128 pixel tiles at 10m resolution
    'tile_size': 128,                   # Pixels per tile side
    'pixel_size': 10,                   # Meters per pixel (Sentinel-1 GRD resolution)

    # ----- Normalization Parameters -----
    # STURM-Flood uses min-max normalization with empirical dB range
    # Formula: normalized = (dB_value - db_min) / (db_max - db_min)
    # Values outside this range will be <0 or >1 (not clipped, matching STURM)
    'db_min': -30,                      # Minimum dB value (maps to 0)
    'db_max': 10,                       # Maximum dB value (maps to 1)

    # ----- Tile Filtering Thresholds -----
    # These control which tiles are included in the final dataset
    'exclude_100pct_lake': False,        # Remove tiles that are 100% lake
    'exclude_100pct_water': False,       # Remove tiles that are 100% any water type
    'min_water_pct': 0,                 # Minimum water % to include (0 = keep all)
    'max_nodata_pct': 0,                # Maximum nodata % allowed (0 = fully within AOI)

    # ----- Vector Filtering (optional, area-specific) -----
    # Some EMS monitoring products include BOTH the current flooded area AND
    # a previous flooded area from an earlier acquisition (e.g. EMSR427 AOI07
    # Kristianstad: dmg_src_id=3 is "Previous Flooded Area" 25/02/2020 05:14,
    # dmg_src_id=4 is "Flooded Area" 28/02/2020 16:52). To keep only polygons
    # that match the Sentinel-1 acquisition date, set an attribute filter
    # where the key is a field name in observedEventA and the value is a list
    # of allowed values. Set to None to disable (default for other areas).
    'observed_event_filter': {'dmg_src_id': [6, 7]},      # Kristianstad: {'dmg_src_id': [4]}

    # ----- AOI Exclusion Mask (optional, area-specific) -----
    # Path to a user-provided shapefile whose polygons are subtracted from
    # the AOI before processing. Use this when the AOI covers areas that are
    # NOT represented in the EMS vector data (e.g. EMSR427 AOI07 Kristianstad
    # extends into the sea but the coastline / Open Water polygon is missing
    # from hydrographyA — without this mask, sea pixels would be labelled as
    # background (class 0), which would inflate false positives during
    # evaluation since both RF and U-Net correctly detect sea as water).
    # Create the mask in QGIS by digitising the sea (or any other region to
    # exclude) as a polygon shapefile in any CRS — it will be reprojected
    # to target_crs automatically. Set to None to disable.
    'aoi_exclusion_mask': None,         # Kristianstad: '/content/uploads/ems_vectors/kristianstad_sea_mask.shp'
}

# ============================================================================
# CLASS MAPPING - Maps EMS attribute values to STURM-Flood class numbers
# ============================================================================
#
# EMS uses descriptive attribute values in their shapefiles.
# We need to convert these to numeric class IDs that match STURM-Flood.
#
# STURM-Flood classes:
#   0 = Background (land, no water)
#   1 = Flooded areas (the actual flood extent we want to detect)
#   2 = Rivers and streams (permanent linear water features)
#   3 = Open water (sea/ocean - mainly for coastal flood events)
#   4 = Reservoirs (man-made water storage)
#   5 = Lakes (permanent polygon water features)
#  99 = No data (outside the analyzed area)

CLASS_MAPPING = {
    # Flooded areas from observedEventA layer
    '5-Flood': 1,

    # Rivers and streams from hydrographyA/L layers
    'BH140-River': 2,
    'BH141-Stream': 2,

    # Open water (sea/ocean) - important for coastal flood events
    # Even if not present in current event, including it won't cause issues
    'BA040-Open Water': 3,

    # Reservoirs (man-made water storage)
    'BH130-Reservoir': 4,

    # Lakes from hydrographyA layer
    'BH080-Lake': 5,

    # EXCLUDED - these are NOT actual flooding:
    # 'BH090-Land Subject to Inundation': 1,  # Risk zone, not observed flooding
}

print("✓ Configuration set!")
print(f"  Event: {CONFIG['ems_code']} - {CONFIG['country']}")
print(f"  Tile size: {CONFIG['tile_size']}x{CONFIG['tile_size']} pixels")
print(f"  Target CRS: {CONFIG['target_crs']}")
print(f"  Max nodata: {CONFIG['max_nodata_pct']}%")

## 3. Create Folder Structure

Create the directory structure that matches STURM-Flood dataset format.

In [ ]:
# Create folder structure matching STURM-Flood dataset organization
#
# /content/
# ├── uploads/
# │   ├── radar/           <- Input: Preprocessed Sentinel-1 GeoTIFF
# │   └── ems_vectors/     <- Input: EMS shapefiles (extracted from ZIP)
# ├── Dataset/
# │   └── Sentinel1/
# │       ├── S1/          <- Output: Normalized radar tiles (128x128)
# │       └── Floodmaps/   <- Output: Ground truth mask tiles (128x128)
# └── temp/                <- Intermediate files (clipped raster, etc.)

folders = [
    '/content/uploads/radar',           # For uploaded radar GeoTIFF
    '/content/uploads/ems_vectors',     # For extracted EMS shapefiles
    '/content/Dataset/Sentinel1/S1',    # Output: normalized radar tiles
    '/content/Dataset/Sentinel1/Floodmaps',  # Output: ground truth tiles
    '/content/temp'                     # Temporary/intermediate files
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

print("\n✓ Folder structure ready!")

## 4. Mount Google Drive and Select Radar File

Connect to Google Drive and use the dropdown to select your preprocessed Sentinel-1 GeoTIFF.

**Requirements for the radar file:**
- Preprocessed in SNAP: Thermal Noise → Orbit → Calibration → Terrain Correction → Speckle Filter → dB
- 2 bands: VV and VH polarization
- Values in dB scale (typically -30 to +10 dB)
- Projected to UTM (matching target_crs in config)

In [ ]:
# Mount Google Drive to access files stored there
# This will prompt you to authorize access to your Drive

drive.mount('/content/drive')
print("✓ Google Drive mounted!")

In [ ]:
# Create interactive file picker for radar GeoTIFF
#
# This scans your Google Drive for .tif files and presents them in a dropdown.
# Select your preprocessed Sentinel-1 file and click the button to load it.

def find_tif_files(folder_path, max_depth=3):
    """
    Recursively search for GeoTIFF files in Google Drive.

    Args:
        folder_path: Root folder to search
        max_depth: Maximum folder depth to search (prevents slow scanning)

    Returns:
        List of (display_name, full_path) tuples
    """
    tif_files = []
    for root, dirs, files_list in os.walk(folder_path):
        depth = root.replace(folder_path, '').count(os.sep)
        if depth < max_depth:
            for f in files_list:
                if f.lower().endswith(('.tif', '.tiff')):
                    full_path = os.path.join(root, f)
                    rel_path = full_path.replace('/content/drive/MyDrive/', '')
                    tif_files.append((rel_path, full_path))
    return sorted(tif_files)

# Scan Drive for GeoTIFF files
print(f"Scanning {BASE} for GeoTIFF files...")
drive_base = '/content/drive/MyDrive/'
tif_files = find_tif_files(BASE)    # only scan BASE-map

if not tif_files:
    print("⚠ No .tif files found in Drive.")
    print("  Upload your preprocessed radar file to Drive and re-run this cell.")
else:
    print(f"Found {len(tif_files)} GeoTIFF files\n")

    # Create dropdown widget for file selection
    dropdown = widgets.Dropdown(
        options=[(name, path) for name, path in tif_files],
        description='Radar file:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )

    # Create button to confirm selection
    button = widgets.Button(
        description="Select this file",
        button_style='primary',
        icon='check'
    )

    output = widgets.Output()
    selected_radar = {'path': None}  # Store selected path

    def on_button_click(b):
        """Handle button click - copy selected file to working directory."""
        with output:
            clear_output()
            selected_radar['path'] = dropdown.value

            # Copy file to uploads folder
            dest = '/content/uploads/radar/' + os.path.basename(dropdown.value)
            print(f"Copying: {os.path.basename(dropdown.value)}")
            shutil.copy(dropdown.value, dest)

            # Verify the file and display info
            with rasterio.open(dest) as src:
                print(f"\n✓ Radar file loaded!")
                print(f"  Size: {src.width} x {src.height} pixels")
                print(f"  Bands: {src.count} (expected: 2 for VV+VH)")
                print(f"  CRS: {src.crs}")
                print(f"  Pixel size: {src.transform[0]:.1f} m")

    button.on_click(on_button_click)
    display(dropdown, button, output)

## 5. Select EMS Vector Data

Use the dropdown to select the EMS vector ZIP file from your Google Drive.

**Download from:** https://mapping.emergency.copernicus.eu/activations/

Look for the "Vector package" download — it contains all the shapefiles.

In [ ]:
# @title
# Select EMS vector ZIP file from Google Drive
#
# The ZIP file from Copernicus EMS contains multiple shapefiles:
# - observedEventA: Observed flood extent polygons (main target!)
# - hydrographyA: Permanent water bodies (lakes, rivers as polygons)
# - hydrographyL: Water features as lines (rivers, streams)
# - areaOfInterestA: The AOI boundary polygon
# - imageFootprintA: Analyzed vs not-analyzed areas

print(f"Scanning {BASE} for EMS vector ZIP files...")
print(f"(Download from: https://emergency.copernicus.eu/mapping/list-of-components/{CONFIG['ems_code']})\n")

def find_zip_files(folder_path, max_depth=4):
    """Recursively search for ZIP files in Google Drive."""
    zip_files = []
    for root, dirs, files_list in os.walk(folder_path):
        depth = root.replace(folder_path, '').count(os.sep)
        if depth < max_depth:
            for f in files_list:
                if f.lower().endswith('.zip'):
                    full_path = os.path.join(root, f)
                    rel_path = full_path.replace('/content/drive/MyDrive/', '')
                    zip_files.append((rel_path, full_path))
    return sorted(zip_files)

zip_files = find_zip_files(BASE)

if not zip_files:
    print("⚠ No .zip files found in Drive.")
    print("  Upload your EMS vector ZIP to Drive and re-run this cell.")
else:
    print(f"Found {len(zip_files)} ZIP files\n")

    dropdown_zip = widgets.Dropdown(
        options=[(name, path) for name, path in zip_files],
        description='EMS ZIP:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )

    button_zip = widgets.Button(
        description="Select this file",
        button_style='primary',
        icon='check'
    )

    output_zip = widgets.Output()

    def on_zip_button_click(b):
        """Handle button click - copy and extract selected ZIP from Drive."""
        with output_zip:
            clear_output()
            zip_path = dropdown_zip.value
            filename = os.path.basename(zip_path)
            dest = f'/content/uploads/ems_vectors/{filename}'

            print(f"Copying: {filename}")
            shutil.copy(zip_path, dest)

            # Extract ZIP file
            print(f"Extracting {filename}...")
            with zipfile.ZipFile(dest, 'r') as zip_ref:
                zip_ref.extractall('/content/uploads/ems_vectors/')
            os.remove(dest)  # Remove ZIP after extraction

            # List all shapefiles found
            shp_files = glob.glob('/content/uploads/ems_vectors/**/*.shp', recursive=True)
            print(f"\n✓ Found {len(shp_files)} shapefiles:")
            for shp in shp_files:
                print(f"  - {os.path.basename(shp)}")

    button_zip.on_click(on_zip_button_click)
    display(dropdown_zip, button_zip, output_zip)


## 6. Helper Functions

These functions handle the core processing steps:
- Finding shapefiles by pattern
- Preparing clean AOI (removing not-analyzed areas)
- Clipping raster to AOI
- Creating ground truth raster from vectors
- Normalizing radar data
- Validating and creating tiles

In [ ]:
def find_shapefile(pattern, search_dir):
    """
    Find a shapefile matching a pattern in a directory.

    Supports both camelCase (EMSR427 style, e.g. 'areaOfInterestA')
    and underscore naming (EMSR280 style, e.g. 'area_of_interest_a').
    The pattern is first searched as-is, then converted to underscore
    format as a fallback — so both EMS naming conventions work automatically.

    Args:
        pattern: Part of filename to match (e.g., 'observedEventA')
        search_dir: Directory to search recursively

    Returns:
        Full path to first matching shapefile, or None if not found
    """
    import re
    # Convert camelCase to underscore: areaOfInterestA -> area_of_interest_a
    underscore = re.sub(r'(?<!^)(?=[A-Z])', '_', pattern).lower()

    for p in [pattern, underscore]:
        matches = glob.glob(f'{search_dir}/**/*{p}*.shp', recursive=True)
        if matches:
            return matches[0]
    return None


def prepare_clean_aoi(ems_dir, target_crs, config=None):
    """
    Load the Area of Interest and remove any 'Not Analysed' zones.

    EMS products sometimes have areas that weren't analyzed (e.g., clouds,
    no satellite coverage). We need to exclude these from our ground truth
    because we don't know what's there.

    The imageFootprintA shapefile contains polygons with obj_type:
    - 'Image Footprint': Area that WAS analyzed (good)
    - 'Not Analysed - No data': Area that was NOT analyzed (exclude)

    If config['aoi_exclusion_mask'] is set, the polygons in that shapefile
    are also subtracted from the AOI. This is used for areas where the EMS
    vector data is incomplete — e.g. Kristianstad extends into the sea but
    hydrographyA does not contain the coastline, so a user-digitised sea
    mask is subtracted to prevent sea pixels from being labelled as
    background.

    Args:
        ems_dir: Directory containing EMS shapefiles
        target_crs: Target coordinate reference system (e.g., 'EPSG:32633')
        config: Optional configuration dict (uses 'aoi_exclusion_mask')

    Returns:
        GeoDataFrame with clean AOI polygon(s)
    """
    print("\nPreparing AOI...")
    print("="*50)

    # Load the Area of Interest boundary
    aoi_path = find_shapefile('areaOfInterestA', ems_dir)
    if not aoi_path:
        raise FileNotFoundError("AOI shapefile (areaOfInterestA) not found!")

    aoi = gpd.read_file(aoi_path)
    print(f"  Loaded AOI: {len(aoi)} polygon(s)")

    # Reproject to target CRS if needed
    if str(aoi.crs) != target_crs:
        aoi = aoi.to_crs(target_crs)
        print(f"  Reprojected to {target_crs}")

    # Load imageFootprint to check for Not Analysed areas
    footprint_path = find_shapefile('imageFootprintA', ems_dir)
    if footprint_path:
        footprint = gpd.read_file(footprint_path)
        if str(footprint.crs) != target_crs:
            footprint = footprint.to_crs(target_crs)

        # Find polygons marked as 'Not Analysed'
        not_analysed = footprint[
            footprint['obj_type'].str.contains('Not Analysed', case=False, na=False)
        ]

        if len(not_analysed) > 0:
            print(f"\n  ⚠ Found {len(not_analysed)} 'Not Analysed' area(s):")
            for idx, row in not_analysed.iterrows():
                print(f"    - {row['obj_type']}")

            # Subtract Not Analysed areas from AOI using geometric difference
            not_analysed_union = unary_union(not_analysed.geometry)
            original_area = aoi.geometry.area.sum()
            aoi['geometry'] = aoi.geometry.difference(not_analysed_union)
            new_area = aoi.geometry.area.sum()

            removed_pct = 100 * (1 - new_area / original_area)
            print(f"\n  ✓ Removed {removed_pct:.1f}% of AOI area")
        else:
            print("  ✓ No 'Not Analysed' areas found - AOI is complete")

    # Optional: subtract a user-provided exclusion mask (area-specific)
    # Used when EMS vectors are incomplete, e.g. sea inside AOI but no
    # coastline polygon in hydrographyA (Kristianstad AOI07).
    if config is not None:
        exclusion_path = config.get('aoi_exclusion_mask')
        if exclusion_path:
            if not os.path.exists(exclusion_path):
                raise FileNotFoundError(
                    f"aoi_exclusion_mask not found: {exclusion_path}"
                )
            print(f"\n  Applying exclusion mask: {os.path.basename(exclusion_path)}")
            mask_gdf = gpd.read_file(exclusion_path)
            if str(mask_gdf.crs) != target_crs:
                mask_gdf = mask_gdf.to_crs(target_crs)

            mask_union = unary_union(mask_gdf.geometry)
            area_before = aoi.geometry.area.sum()
            aoi['geometry'] = aoi.geometry.difference(mask_union)
            area_after = aoi.geometry.area.sum()

            removed_pct = 100 * (1 - area_after / area_before)
            print(f"  ✓ Removed {removed_pct:.1f}% of AOI area via exclusion mask")

    # Save clean AOI for reference/debugging
    aoi.to_file('/content/temp/clean_aoi.shp')
    print(f"\n  ✓ Clean AOI saved to /content/temp/clean_aoi.shp")

    return aoi


def clip_raster_to_aoi(raster_path, aoi_gdf, output_path):
    """
    Clip a raster to the AOI polygon boundary.

    This clips the raster to the AOI extent. Pixels outside the AOI
    are set to nodata (-9999) in the clipped raster, but these are
    eliminated in the tiling step which only keeps tiles fully within the AOI.

    Args:
        raster_path: Path to input GeoTIFF
        aoi_gdf: GeoDataFrame with AOI polygon(s)
        output_path: Path for output clipped GeoTIFF

    Returns:
        Path to the clipped raster
    """
    print("\nClipping raster to AOI...")
    print("="*50)

    with rasterio.open(raster_path) as src:
        print(f"  Input size: {src.width} x {src.height} pixels")

        # Ensure AOI is in same CRS as raster
        if aoi_gdf.crs != src.crs:
            aoi_gdf = aoi_gdf.to_crs(src.crs)

        # Clip raster using AOI geometry
        # crop=True shrinks the raster extent to the AOI bounding box
        # nodata=-9999 marks pixels outside the polygon
        clipped, transform = mask(
            src,
            aoi_gdf.geometry,
            crop=True,
            nodata=-9999,
            all_touched=True  # Include pixels that touch the boundary
        )

        # Update raster metadata for the new size
        profile = src.profile.copy()
        profile.update(
            height=clipped.shape[1],
            width=clipped.shape[2],
            transform=transform,
            nodata=-9999
        )

        # Write clipped raster
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(clipped)

        print(f"  Output size: {clipped.shape[2]} x {clipped.shape[1]} pixels")
        reduction = 100 * (1 - (clipped.shape[1] * clipped.shape[2]) / (src.height * src.width))
        print(f"  Size reduction: {reduction:.1f}%")
        print(f"  ✓ Saved to {output_path}")

    return output_path


def load_and_classify_vectors(ems_dir, target_crs, config=None):
    """
    Load EMS vector layers and assign STURM-Flood class numbers.

    Reads the observedEventA (floods) and hydrographyA/L (water features)
    shapefiles and maps their attribute values to numeric class IDs.

    If config contains 'observed_event_filter', only features in the
    observedEventA layer whose attribute values match the filter are kept.
    This is used e.g. for EMSR427 AOI07 (Kristianstad) to exclude the
    Previous Flooded Area polygons that don't match the Sentinel-1 date.

    Args:
        ems_dir: Directory containing EMS shapefiles
        target_crs: Target coordinate reference system
        config: Optional configuration dict (uses 'observed_event_filter')

    Returns:
        GeoDataFrame with 'geometry' and 'class' columns
    """
    all_features = []

    # Define which files to load and which attribute contains the class info
    files_config = {
        'observedEventA': ('observedEventA', 'event_type'),  # Flood polygons
        'hydrographyA': ('hydrographyA', 'obj_type'),        # Water area polygons
        'hydrographyL': ('hydrographyL', 'obj_type'),        # Water lines (rivers)
    }

    for layer_name, (pattern, attr_field) in files_config.items():
        filepath = find_shapefile(pattern, ems_dir)

        if not filepath:
            print(f"  Warning: {pattern} not found - skipping")
            continue

        print(f"  Loading {os.path.basename(filepath)}...")
        gdf = gpd.read_file(filepath)

        # Reproject if needed
        if str(gdf.crs) != target_crs:
            gdf = gdf.to_crs(target_crs)

        # Optional: filter observedEventA by attribute values (area-specific)
        # e.g. Kristianstad: keep only dmg_src_id=4 (current flood, matches Sentinel-1 date)
        if layer_name == 'observedEventA' and config is not None:
            obs_filter = config.get('observed_event_filter')
            if obs_filter:
                for field, allowed in obs_filter.items():
                    if field in gdf.columns:
                        before = len(gdf)
                        gdf = gdf[gdf[field].isin(allowed)].copy()
                        print(f"    Filter: {field} ∈ {allowed} "
                              f"→ {before} → {len(gdf)} features")
                    else:
                        print(f"    ⚠ Filter field '{field}' not found in {layer_name} "
                              f"(columns: {list(gdf.columns)})")

        # Map attribute values to class numbers using CLASS_MAPPING
        gdf['class'] = gdf[attr_field].map(CLASS_MAPPING)

        # Keep only features that mapped to a valid class (>0)
        gdf = gdf[gdf['class'].notna() & (gdf['class'] > 0)]

        if len(gdf) > 0:
            print(f"    → {len(gdf)} features mapped to classes")
            all_features.append(gdf[['geometry', 'class']])

    if not all_features:
        raise ValueError("No features found! Check CLASS_MAPPING.")

    # Combine all features into single GeoDataFrame
    combined = gpd.GeoDataFrame(pd.concat(all_features, ignore_index=True))
    combined['class'] = combined['class'].astype(int)
    combined = combined.set_crs(target_crs)

    return combined


def create_ground_truth_raster(ems_dir, reference_raster, aoi_gdf, output_path, config):
    """
    Create ground truth raster by rasterizing EMS vector polygons.

    The raster will have the same extent and resolution as the reference raster
    (the clipped radar image). Each pixel gets a class value:
    - 0: Background (land)
    - 1: Flooded area
    - 2: River/stream
    - 3: Open water (sea)
    - 5: Lake
    - 99: Outside AOI (no data)

    Args:
        ems_dir: Directory containing EMS shapefiles
        reference_raster: Path to raster defining output extent/resolution
        aoi_gdf: GeoDataFrame with AOI polygon (for masking)
        output_path: Path for output ground truth GeoTIFF
        config: Configuration dictionary

    Returns:
        The ground truth raster as numpy array
    """
    print("\nCreating ground truth raster...")
    print("="*50)

    target_crs = config['target_crs']

    # Step 1: Load and classify vector features
    print("\n1. Loading vector data...")
    features_gdf = load_and_classify_vectors(ems_dir, target_crs, config=config)
    print(f"   Total features to rasterize: {len(features_gdf)}")

    # Show class distribution
    class_names = {1: 'Flooded', 2: 'River/Stream', 3: 'Open Water', 5: 'Lake'}
    for cls in sorted(features_gdf['class'].unique()):
        count = len(features_gdf[features_gdf['class'] == cls])
        name = class_names.get(cls, f'Class {cls}')
        print(f"   - {name}: {count} features")

    # Step 2: Get raster parameters from reference
    print("\n2. Reading reference raster parameters...")
    with rasterio.open(reference_raster) as src:
        transform = src.transform
        width = src.width
        height = src.height
        crs = src.crs
    print(f"   Output size: {width} x {height} pixels")

    # Step 3: Rasterize vector features
    # Sort by class (descending) so floods overwrite permanent water
    # This means if a flood polygon overlaps a lake, we keep the flood label
    print("\n3. Rasterizing features...")
    features_gdf_sorted = features_gdf.sort_values('class', ascending=False)
    shapes = [(geom, value) for geom, value in
              zip(features_gdf_sorted.geometry, features_gdf_sorted['class'])]

    raster = features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,              # Background class
        dtype=np.uint8,
        all_touched=True     # Include pixels touched by polygon edges
    )

    # Step 4: Apply AOI mask (set pixels outside AOI to nodata=99)
    print("\n4. Applying AOI mask...")
    if aoi_gdf.crs != crs:
        aoi_gdf = aoi_gdf.to_crs(crs)

    aoi_mask = features.rasterize(
        [(geom, 1) for geom in aoi_gdf.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    # Pixels outside AOI become class 99 (no data)
    raster = np.where(aoi_mask == 1, raster, 99)

    # Step 5: Print statistics
    print("\n5. Ground truth statistics:")
    unique, counts = np.unique(raster, return_counts=True)
    class_names_full = {
        0: "Background", 1: "Flooded", 2: "River/Stream",
        3: "Open Water", 5: "Lake", 99: "No Data"
    }
    for val, count in zip(unique, counts):
        pct = 100 * count / raster.size
        name = class_names_full.get(val, f"Class {val}")
        print(f"   {name} ({val}): {count:,} pixels ({pct:.2f}%)")

    # Step 6: Save raster
    print(f"\n6. Saving to {output_path}...")
    profile = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'width': width,
        'height': height,
        'count': 1,
        'crs': crs,
        'transform': transform,
        'nodata': 99
    }

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(raster, 1)

    print("\n✓ Ground truth raster created!")
    return raster


def normalize_radar(data, db_min=-30, db_max=10):
    """
    Normalize radar dB values using STURM-Flood formula.

    STURM-Flood uses min-max normalization based on empirical dB range.
    The formula maps -30 dB → 0 and +10 dB → 1.

    Note: Values outside this range are NOT clipped, matching STURM behavior.
    A pixel with -40 dB would become -0.25, and +15 dB would become 1.125.

    Args:
        data: Radar data in dB scale (numpy array)
        db_min: Minimum dB value (maps to 0)
        db_max: Maximum dB value (maps to 1)

    Returns:
        Normalized data (typically 0-1, but can exceed this range)
    """
    return (data - db_min) / (db_max - db_min)


def check_tile_validity(gt_tile, config):
    """
    Check if a tile should be included in the dataset.

    Tiles are excluded if they:
    - Have too much nodata (outside AOI)
    - Are 100% lake (no useful training signal)
    - Are 100% open water/sea (no useful training signal)

    Args:
        gt_tile: Ground truth tile (128x128 numpy array)
        config: Configuration dictionary with filtering thresholds

    Returns:
        Tuple of (is_valid, reason, stats_dict)
    """
    total_pixels = gt_tile.size  # 128 * 128 = 16384

    # Count pixels in each class
    unique, counts = np.unique(gt_tile, return_counts=True)
    class_counts = dict(zip(unique, counts))

    # Calculate pixel counts for each category
    nodata_count = class_counts.get(99, 0)       # Outside AOI
    flood_count = class_counts.get(1, 0)        # Flooded areas
    river_count = class_counts.get(2, 0)        # Rivers/streams
    open_water_count = class_counts.get(3, 0)   # Sea/ocean
    reservoir_count = class_counts.get(4, 0)    # Reservoirs
    lake_count = class_counts.get(5, 0)         # Lakes

    # Total water = all water classes combined
    water_count = flood_count + river_count + open_water_count + reservoir_count + lake_count

    # Convert to percentages
    nodata_pct = 100 * nodata_count / total_pixels
    flood_pct = 100 * flood_count / total_pixels
    river_pct = 100 * river_count / total_pixels
    open_water_pct = 100 * open_water_count / total_pixels
    reservoir_pct = 100 * reservoir_count / total_pixels
    lake_pct = 100 * lake_count / total_pixels
    water_pct = 100 * water_count / total_pixels

    # Compile statistics for metadata
    stats = {
        'water_pct': water_pct,
        'flood_pct': flood_pct,
        'river_pct': river_pct,
        'open_water_pct': open_water_pct,
        'reservoir_pct': reservoir_pct,
        'lake_pct': lake_pct,
        'nodata_pct': nodata_pct,
    }

    # ----- Apply exclusion rules -----

    # Rule 1: Exclude tiles with too much nodata (outside AOI)
    if nodata_pct > config['max_nodata_pct']:
        return False, f"Too much nodata ({nodata_pct:.1f}%)", stats

    # Rule 2: Exclude tiles that are 100% lake
    if config.get('exclude_100pct_lake', True) and lake_pct >= 99.9:
        return False, "100% lake", stats

    # Rule 3: Exclude tiles that are 100% open water (sea/ocean)
    if config.get('exclude_100pct_water', True) and open_water_pct >= 99.9:
        return False, "100% open water", stats

    # Rule 4: Exclude tiles that are 100% any water type
    if config.get('exclude_100pct_water', True) and water_pct >= 99.9:
        return False, "100% water", stats

    return True, "OK", stats


def create_tiles(radar_path, gt_path, output_dir, config):
    """
    Create 128x128 pixel tiles from radar and ground truth rasters.

    This is the main tiling function that:
    1. Opens both rasters
    2. Iterates through a grid of tile positions
    3. For each position, checks if the tile is valid
    4. If valid, normalizes radar data and saves both tiles
    5. Records metadata for each tile

    Args:
        radar_path: Path to clipped radar GeoTIFF
        gt_path: Path to ground truth GeoTIFF
        output_dir: Base directory for output (e.g., '/content/Dataset')
        config: Configuration dictionary

    Returns:
        DataFrame with metadata for all valid tiles
    """
    print("\nCreating tiles...")
    print("="*50)

    tile_size = config['tile_size']  # 128 pixels
    s1_dir = f"{output_dir}/Sentinel1/S1"           # Radar tiles
    gt_dir = f"{output_dir}/Sentinel1/Floodmaps"    # Ground truth tiles

    metadata_rows = []  # Will become the metadata CSV

    with rasterio.open(radar_path) as radar_src, rasterio.open(gt_path) as gt_src:
        # Verify dimensions match
        assert radar_src.width == gt_src.width, "Width mismatch between radar and GT!"
        assert radar_src.height == gt_src.height, "Height mismatch between radar and GT!"

        height, width = radar_src.height, radar_src.width

        # Calculate tile grid
        # We only use complete tiles (edges are discarded)
        n_tiles_y = height // tile_size
        n_tiles_x = width // tile_size
        total_tiles = n_tiles_x * n_tiles_y

        # Calculate how many pixels we lose at edges
        lost_y = height % tile_size
        lost_x = width % tile_size

        print(f"Raster size: {width} x {height} pixels")
        print(f"Tile size: {tile_size} x {tile_size} pixels")
        print(f"Tile grid: {n_tiles_x} x {n_tiles_y} = {total_tiles} potential tiles")
        print(f"Edge pixels discarded: {lost_x} (x) and {lost_y} (y)")
        print(f"\nProcessing tiles...")

        valid_count = 0
        excluded_reasons = {}  # Track why tiles were excluded

        # Iterate through tile grid
        for y in range(n_tiles_y):
            for x in range(n_tiles_x):
                # Define the window (subset) to read
                window = Window(
                    col_off=x * tile_size,
                    row_off=y * tile_size,
                    width=tile_size,
                    height=tile_size
                )

                # Read ground truth tile and check validity
                gt_tile = gt_src.read(1, window=window)
                is_valid, reason, stats = check_tile_validity(gt_tile, config)

                if not is_valid:
                    # Track exclusion reason
                    excluded_reasons[reason] = excluded_reasons.get(reason, 0) + 1
                    continue

                # Read and normalize radar tile
                radar_tile = radar_src.read(window=window).astype(np.float32)
                radar_tile_norm = normalize_radar(
                    radar_tile,
                    db_min=config['db_min'],
                    db_max=config['db_max']
                )

                # Generate tile filename
                tile_id = f"{config['ems_code']}_{config['aoi_code']}_{x}_{y}.tif"

                # Calculate tile's geotransform
                tile_transform = rasterio.windows.transform(window, radar_src.transform)

                # ----- Save radar tile -----
                radar_profile = radar_src.profile.copy()
                radar_profile.update(
                    height=tile_size,
                    width=tile_size,
                    transform=tile_transform,
                    dtype='float32',
                    nodata=None  # Remove inherited nodata value
                )

                with rasterio.open(f"{s1_dir}/{tile_id}", 'w', **radar_profile) as dst:
                    dst.write(radar_tile_norm)

                # ----- Save ground truth tile -----
                gt_profile = gt_src.profile.copy()
                gt_profile.update(
                    height=tile_size,
                    width=tile_size,
                    transform=tile_transform,
                    nodata=None  # Remove inherited nodata value
                )

                with rasterio.open(f"{gt_dir}/{tile_id}", 'w', **gt_profile) as dst:
                    dst.write(gt_tile, 1)

                # ----- Record metadata -----
                metadata_rows.append({
                    'tile_id': tile_id,
                    'ems_code': config['ems_code'],
                    'aoi_code': config['aoi_code'],
                    'grid_x': x,
                    'grid_y': y,
                    'water_pct': stats['water_pct'],
                    'flood_pct': stats['flood_pct'],
                    'lake_pct': stats['lake_pct'],
                    'river_pct': stats['river_pct'],
                    'open_water_pct': stats['open_water_pct'],
                    'reservoir_pct': stats['reservoir_pct'],
                    'nodata_pct': stats['nodata_pct'],
                })

                valid_count += 1

        # Print summary
        print(f"\n✓ Created {valid_count} valid tiles")
        print(f"\nExcluded tiles by reason:")
        for reason, count in sorted(excluded_reasons.items(), key=lambda x: -x[1]):
            print(f"  - {reason}: {count}")

    return pd.DataFrame(metadata_rows)

## 7. Run Processing Pipeline

Execute the full processing workflow step by step.

In [ ]:
# Step 1: Prepare clean AOI (remove "Not Analysed" areas)
aoi = prepare_clean_aoi('/content/uploads/ems_vectors', CONFIG['target_crs'], config=CONFIG)

In [ ]:
# Step 2: Clip radar image to the clean AOI boundary
radar_input = glob.glob('/content/uploads/radar/*.tif')[0]
radar_clipped = '/content/temp/radar_clipped.tif'

clip_raster_to_aoi(radar_input, aoi, radar_clipped)

In [ ]:
# Step 3: Create ground truth raster from EMS vectors
gt_path = '/content/temp/ground_truth.tif'

gt_raster = create_ground_truth_raster(
    ems_dir='/content/uploads/ems_vectors',
    reference_raster=radar_clipped,
    aoi_gdf=aoi,
    output_path=gt_path,
    config=CONFIG
)

## 7b. Create Output Folders on Drive

Create the required folder structure in Google Drive before writing tiles.

In [ ]:
import os
for folder in [
    f'{OUTPUT_DATASET_DIR}/Sentinel1/S1',
    f'{OUTPUT_DATASET_DIR}/Sentinel1/Floodmaps',
]:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

In [ ]:
# Step 4: Normalize radar and create 128x128 tiles
metadata_df = create_tiles(
    radar_path=radar_clipped,
    gt_path=gt_path,
    output_dir=OUTPUT_DATASET_DIR,
    config=CONFIG
)

# Save metadata CSV
metadata_path = f"{OUTPUT_DATASET_DIR}/{CONFIG['ems_code']}_{CONFIG['aoi_code']}_metadata.csv"
metadata_df.to_csv(metadata_path, index=False)
print(f"\n✓ Metadata saved to {metadata_path}")
print(f"  Total tiles: {len(metadata_df)}")

## 8. Dataset Summary

Review statistics about the created dataset.

In [ ]:
# Display dataset summary statistics
print("="*60)
print("DATASET SUMMARY")
print("="*60)

print(f"\nTotal tiles: {len(metadata_df)}")
print(f"Tile size: {CONFIG['tile_size']}x{CONFIG['tile_size']} pixels ({CONFIG['tile_size'] * CONFIG['pixel_size']}m x {CONFIG['tile_size'] * CONFIG['pixel_size']}m)")

print("\n--- Water Coverage Distribution ---")
print(metadata_df[['water_pct', 'flood_pct', 'lake_pct', 'river_pct', 'open_water_pct', 'reservoir_pct']].describe().round(2))

print("\n--- Tile Categories ---")
print(f"Tiles with flood: {len(metadata_df[metadata_df['flood_pct'] > 0])} ({100*len(metadata_df[metadata_df['flood_pct'] > 0])/len(metadata_df):.1f}%)")
print(f"Tiles with lake: {len(metadata_df[metadata_df['lake_pct'] > 0])} ({100*len(metadata_df[metadata_df['lake_pct'] > 0])/len(metadata_df):.1f}%)")
print(f"Tiles with river: {len(metadata_df[metadata_df['river_pct'] > 0])} ({100*len(metadata_df[metadata_df['river_pct'] > 0])/len(metadata_df):.1f}%)")
print(f"Tiles with reservoir: {len(metadata_df[metadata_df['reservoir_pct'] > 0])} ({100*len(metadata_df[metadata_df['reservoir_pct'] > 0])/len(metadata_df):.1f}%)")
print(f"Tiles with any water: {len(metadata_df[metadata_df['water_pct'] > 0])} ({100*len(metadata_df[metadata_df['water_pct'] > 0])/len(metadata_df):.1f}%)")
print(f"Tiles with no water (land only): {len(metadata_df[metadata_df['water_pct'] == 0])} ({100*len(metadata_df[metadata_df['water_pct'] == 0])/len(metadata_df):.1f}%)")

## 9. Visualize Sample Tile

Preview a sample tile with water content — VV band, VH band, and ground truth mask.


In [ ]:
# @title
# Visualize a sample tile with water content
import matplotlib.pyplot as plt

# Find a tile with some water to visualize
# Change iloc[N] below to view a different tile (0-indexed)
if len(metadata_df[metadata_df['water_pct'] > 5]) > 0:
    sample = metadata_df[metadata_df['water_pct'] > 5].iloc[10]
else:
    sample = metadata_df.iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Load radar tile
with rasterio.open(f"{OUTPUT_DATASET_DIR}/Sentinel1/S1/{sample['tile_id']}") as src:
    vv = src.read(1)
    vh = src.read(2)

# Plot VV band (normalized)
axes[0].imshow(vv, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f"VV (normalized)\n{sample['tile_id']}")
axes[0].axis('off')

# Plot VH band (normalized)
axes[1].imshow(vh, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('VH (normalized)')
axes[1].axis('off')

# Load and plot ground truth
with rasterio.open(f"{OUTPUT_DATASET_DIR}/Sentinel1/Floodmaps/{sample['tile_id']}") as src:
    gt = src.read(1)

# Create colormap for classes
from matplotlib.colors import ListedColormap
# Colors: 0=green(land), 1=red(flood), 2=blue(river), 3=cyan(sea), 4=purple, 5=darkblue(lake)
colors = ['#228B22', '#FF6B6B', '#4169E1', '#00CED1', '#9370DB', '#000080']
cmap = ListedColormap(colors)

im = axes[2].imshow(gt, cmap=cmap, vmin=0, vmax=5)
axes[2].set_title(f"Ground Truth\nWater: {sample['water_pct']:.1f}%, Flood: {sample['flood_pct']:.1f}%")
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/Dataset/sample_preview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Preview saved to /content/Dataset/sample_preview.png")

## 10. Tile Coverage Map

Interactive map of all tiles in the dataset. Each tile is colored by its ground truth water percentage; click a tile for details.

In [ ]:
import folium
import rasterio
import os
from pyproj import Transformer

# Point to the prepared tiles for the configured study area
tiles_dir = f'{OUTPUT_DATASET_DIR}/Sentinel1/S1'
tiles = [f for f in os.listdir(tiles_dir) if f.endswith('.tif')]

# Get tile bounds and transform to lat/lon
transformer = Transformer.from_crs(CONFIG['target_crs'], 'EPSG:4326', always_xy=True)

bounds_list = []
for t in tiles:
    with rasterio.open(os.path.join(tiles_dir, t)) as src:
        b = src.bounds
        lon_min, lat_min = transformer.transform(b.left, b.bottom)
        lon_max, lat_max = transformer.transform(b.right, b.top)
        bounds_list.append((t, lat_min, lat_max, lon_min, lon_max))

# Create map centered on tiles
center_lat = sum(b[1] + b[2] for b in bounds_list) / (2 * len(bounds_list))
center_lon = sum(b[3] + b[4] for b in bounds_list) / (2 * len(bounds_list))
m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

# Read metadata for coloring by water_pct
metadata_path = f"{OUTPUT_DATASET_DIR}/{CONFIG['ems_code']}_{CONFIG['aoi_code']}_metadata.csv"
if os.path.exists(metadata_path):
    import pandas as pd
    meta = pd.read_csv(metadata_path)
    water_lookup = dict(zip(meta['tile_id'], meta['water_pct']))
else:
    water_lookup = {}

# Add tile rectangles
for name, lat_min, lat_max, lon_min, lon_max in bounds_list:
    wpct = water_lookup.get(name, 0)
    # Blue intensity by water percentage
    color = f'#{0:02x}{0:02x}{min(255, int(50 + wpct * 5)):02x}' if wpct > 0 else '#228B22'

    folium.Rectangle(
        bounds=[[lat_min, lon_min], [lat_max, lon_max]],
        color=color,
        fill=True,
        fill_opacity=0.4,
        popup=f"{name}<br>Water: {wpct:.1f}%"
    ).add_to(m)

print(f"✓ {len(tiles)} tiles on map")
m

## Summary

The dataset is now ready for STURM-Flood inference!

**Output structure:**
```
Dataset/
├── Sentinel1/
│   ├── S1/              <- Normalized radar tiles (128x128, 2 bands: VV+VH)
│   └── Floodmaps/       <- Ground truth masks (128x128, class values)
├── {ems_code}_metadata.csv
└── sample_preview.png
```

**Next steps:**
1. The prepared tiles are now in `{OUTPUT_DATASET_DIR}/Sentinel1/`, ready for model inference or training.
2. Review the metadata CSV to inspect tile statistics.
3. Use the folium map above to verify spatial coverage.